# Evaluate All Architectures (Classification Performance)

## 1. Configuration

In [ ]:
from pathlib import Path
import torch

PROJECT_ROOT = Path(r"D:\Ravishi\MSc Final Project\skin-lesion-xai")

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
EVAL_DIR = RESULTS_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

EVAL_CSV = DATA_PROCESSED / "val.csv"

IMAGE_SIZE = 224
BATCH_SIZE = 16
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

MODEL_CONFIGS = {
    "cnn_baseline":  {"timm_name": "resnet50",             "description": "Baseline CNN (ResNet-50)"},
    "attention_cnn": {"timm_name": "resnet50",             "description": "ResNet-50 + CBAM"},
    "vit":           {"timm_name": "deit_small_patch16_224","description": "DeiT-Small (transformer)"},
}
DISPLAY_NAMES = {"cnn_baseline": "CNN", "attention_cnn": "Attention-CNN", "vit": "DeiT"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Evaluating on:", EVAL_CSV.name)


## 2. Model definitions

In [ ]:
import timm
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1); self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(nn.Conv2d(channels, hidden, 1, bias=False),
                                 nn.ReLU(inplace=True), nn.Conv2d(hidden, channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(kernel_size)
    def forward(self, x):
        return self.spatial_attn(self.channel_attn(x))

class AttentionCNN(nn.Module):
    def __init__(self, backbone_name="resnet50", num_classes=2, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, global_pool="")
        feat_dim = self.backbone.num_features
        self.cbam = CBAM(feat_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(feat_dim, num_classes)
    def forward(self, x):
        feats = self.cbam(self.backbone(x))
        return self.fc(self.dropout(self.pool(feats).flatten(1)))

class DropoutCNN(nn.Module):

    def __init__(self, backbone_name="resnet50", num_classes=2, pretrained=False,
                 dropout=0.5):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, global_pool="")
        feat_dim = self.backbone.num_features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        return self.fc(self.dropout(self.pool(self.backbone(x)).flatten(1)))

def build_model(model_key, num_classes=2):
    timm_name = MODEL_CONFIGS[model_key]["timm_name"]
    if model_key == "attention_cnn":
        return AttentionCNN(timm_name, num_classes, pretrained=False)
    if model_key == "cnn_baseline":
        return DropoutCNN(timm_name, num_classes, pretrained=False)
    return timm.create_model(timm_name, pretrained=False, num_classes=num_classes)


def load_trained(model_key):
    ckpt_path = MODELS_DIR / f"{model_key}_best.pth"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    model = build_model(model_key)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt["model_state"] if "model_state" in ckpt else ckpt
    model.load_state_dict(state)
    model.to(device).eval()
    trained_epoch = ckpt.get("epoch", "?")
    print(f"  loaded {model_key} (best epoch {trained_epoch})")
    return model

print("Model definitions ready.")


## 3. Data loader for evaluation (no augmentation)

In [ ]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class EvalDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
        self.tf = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self.tf(Image.open(row["image_path"]).convert("RGB"))
        return img, int(row["label"])

eval_ds = EvalDataset(EVAL_CSV)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Evaluation set: {len(eval_ds)} images")


## 4. Run inference for all three models

In [ ]:
import numpy as np

def predict(model):
    all_true, all_prob = [], []
    with torch.no_grad():
        for images, labels in eval_loader:
            probs = torch.softmax(model(images.to(device)).float(), dim=1)[:, 1]
            all_prob.append(probs.cpu().numpy())
            all_true.append(np.asarray(labels))
    return np.concatenate(all_true), np.concatenate(all_prob)

print("Loading checkpoints and running inference...")
predictions = {}
for key in MODEL_CONFIGS:
    model = load_trained(key)
    y_true, y_prob = predict(model)
    predictions[key] = {"y_true": y_true, "y_prob": y_prob}
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
print("Done.")


## 5. Summary metrics table




In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score,
                             precision_score, f1_score, roc_auc_score, confusion_matrix)

rows = []
for key in MODEL_CONFIGS:
    yt = predictions[key]["y_true"]; yp = predictions[key]["y_prob"]
    pred = (yp >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, pred, labels=[0, 1]).ravel()
    rows.append({
        "Model": DISPLAY_NAMES[key],
        "Accuracy": accuracy_score(yt, pred),
        "Balanced Acc": balanced_accuracy_score(yt, pred),
        "Sensitivity (mel)": recall_score(yt, pred, pos_label=1, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "Precision (mel)": precision_score(yt, pred, pos_label=1, zero_division=0),
        "F1 (mel)": f1_score(yt, pred, pos_label=1, zero_division=0),
        "AUC": roc_auc_score(yt, yp) if len(np.unique(yt)) > 1 else float("nan"),
    })

summary = pd.DataFrame(rows).set_index("Model").round(3)
summary.to_csv(EVAL_DIR / "summary_metrics.csv")
print(summary.to_string())
print(f"\nSaved: {EVAL_DIR / 'summary_metrics.csv'}")
summary


## 6. Confusion matrices (one per architecture)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, key in zip(axes, MODEL_CONFIGS):
    yt = predictions[key]["y_true"]
    pred = (predictions[key]["y_prob"] >= 0.5).astype(int)
    cm = confusion_matrix(yt, pred, labels=[0, 1])
    im = ax.imshow(cm, cmap="Blues")
    names = ["non-mel", "melanoma"]
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(names); ax.set_yticklabels(names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(DISPLAY_NAMES[key])
    thresh = cm.max() / 2 if cm.max() else 0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
fig.suptitle("Confusion matrices (validation)")
plt.tight_layout()
plt.savefig(EVAL_DIR / "confusion_matrices.png", dpi=150)
plt.show()


## 7. ROC curves (all three overlaid)

In [ ]:
from sklearn.metrics import roc_curve, auc

fig, ax = plt.subplots(figsize=(7, 6))
for key in MODEL_CONFIGS:
    yt = predictions[key]["y_true"]; yp = predictions[key]["y_prob"]
    fpr, tpr, _ = roc_curve(yt, yp)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{DISPLAY_NAMES[key]} (AUC = {roc_auc:.3f})", linewidth=2)

ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Chance")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curves — melanoma detection (validation)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(EVAL_DIR / "roc_curves.png", dpi=150)
plt.show()

